# Дообучение эмбеддера `multilingual-e5-small` на датасете Adilet Legal QA

Ноутбук выполняет полный цикл дообучения (fine-tuning) модели поиска для НПА РК:

1. **Разбиение данных** на train / val / test **по фрагментам закона** (`context_id`), со стратификацией по законам.
   Все вопросы одного фрагмента попадают только в одну часть — модель не видит тестовые фрагменты при обучении.
2. **Чанкинг** корпуса (фиксированное окно 500 символов, перекрытие 50) и поиск эталонных чанков для каждого вопроса.
3. **Baseline** — исходная модель без дообучения (Dense, BM25, Hybrid) на val и test.
4. **Hard negatives** — сложные отрицательные примеры из выдачи исходной модели.
5. **Дообучение** контрастной функцией потерь `MultipleNegativesRankingLoss` — несколько запусков с разными seed.
6. **Сравнение «до / после»** на тестовой выборке (среднее ± стандартное отклонение по запускам) и сохранение результатов.

> Перед запуском: `python prepare_adilet_data.py --lang all` (создаёт `data/raw/*.parquet` и `data/adilet/qa_*_full.json`).
> Нужны пакеты `datasets` и `accelerate` (есть в `requirements.txt`). На CPU полное обучение идёт долго — лучше GPU (Colab / Kaggle).
> После ноутбука запустите `python tune_hybrid.py --lang rus`: он подбирает веса гибрида для сохранённой модели на val.

In [1]:
# --- Только для Google Colab / Kaggle: раскомментируйте, чтобы скачать проект и данные ---
# !git clone https://github.com/DamirKx/rag.git
# %cd rag
# !pip install -q -r requirements.txt
# !python prepare_adilet_data.py --lang all

## 0. Настройки

In [2]:
import os
import sys
import gc
import json
import random
import shutil
from collections import defaultdict
from pathlib import Path

# Ноутбук лежит в notebooks/, а модули проекта — в корне репозитория
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from src.embeddings import EmbeddingModel
from src.retriever import VectorRetriever, BM25Retriever, HybridRetriever
from src.eval import calculate_mrr, calculate_hit_rate_at_k, calculate_recall_at_k
from prepare_adilet_data import clean_adilet_context
from run_adilet_eval import GoldChunkMatcher, build_fixed_chunks, get_or_compute_embeddings, model_cache_name

In [3]:
LANG = "rus"                                   # "rus" или "kaz"
BASE_MODEL = "intfloat/multilingual-e5-small"
SEED = 42                   # seed разбиения и выбора hard negatives — общий для всех запусков
TRAIN_SEEDS = [42, 43, 44]  # seed обучения (порядок батчей, dropout); разброс метрик по ним показывает устойчивость

# Доли train / val / test (считаются по фрагментам закона, а не по вопросам)
SPLIT_RATIOS = (0.8, 0.1, 0.1)

# Чанкинг — как в run_adilet_eval.py
CHUNK_SIZE, CHUNK_OVERLAP = 500, 50
MIN_COVERAGE = 0.5        # мин. доля слов ответа в чанке, чтобы считать его эталонным (см. раздел 3)
TIE_MARGIN = 0.05         # чанки, уступающие лучшему по доле слов не больше этого, тоже эталонные

# Hard negatives
NEG_SEARCH_DEPTH = 50     # сколько кандидатов достаём из индекса
NEG_POOL = 10             # негатив выбирается случайно из первых NEG_POOL подходящих
NEG_MAX_RATIO = 0.98      # кандидат отбрасывается, если он похож на вопрос почти как эталон (вероятный ложный негатив).
                          # У e5 сходства сжаты (~0.85–0.9), поэтому порог 0.95 из NV-Retriever отсекает больше половины кандидатов

# Обучение
EPOCHS = 2                # при 3 эпохах MRR@10 на val почти не рос после первой (0.640 → 0.645 → 0.644)
BATCH_SIZE = 32           # для MNRL чем больше батч, тем больше «бесплатных» негативов
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
MAX_SEQ_LEN = 256         # чанк в 500 символов укладывается в ~150–200 токенов

# Быстрая проверка, что всё работает: маленький корпус, 1 эпоха, 2 запуска
SMOKE_TEST = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SPLIT_DIR = Path("data/adilet/splits")
OUTPUT_DIR = Path(f"models/e5-small-adilet-{LANG}")          # сюда копируется лучший по val запуск
CHECKPOINT_DIR = Path(f"checkpoints/e5-small-adilet-{LANG}")  # модели всех запусков: seed_42/, seed_43/, ...
CACHE_DIR = "data/adilet/cache"

if SMOKE_TEST:
    EPOCHS = 1
    TRAIN_SEEDS = TRAIN_SEEDS[:2]
    OUTPUT_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + "-smoke")
    CHECKPOINT_DIR = CHECKPOINT_DIR.with_name(CHECKPOINT_DIR.name + "-smoke")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Устройство: {DEVICE} | язык: {LANG} | модель: {BASE_MODEL} | seeds: {TRAIN_SEEDS} | smoke: {SMOKE_TEST}")

Устройство: cuda | язык: rus | модель: intfloat/multilingual-e5-small | seeds: [42, 43, 44] | smoke: False


## 1. Загрузка данных

In [4]:
df = pd.read_parquet("data/raw/adilet_legal_qa_kz.parquet")
sub_df = df[df["language"] == LANG].copy().reset_index(drop=True)
sub_df["cleaned_context"] = sub_df["context"].apply(clean_adilet_context)
# Идентификаторы совпадают с prepare_adilet_data.py: ctx_0000, ctx_0001, ...
sub_df["context_id"] = [f"ctx_{i:04d}" for i in range(len(sub_df))]

with open(f"data/adilet/qa_{LANG}_full.json", encoding="utf-8") as f:
    qa_all = json.load(f)

# Проверка, что QA-файл построен по тому же parquet
ctx_law = dict(zip(sub_df["context_id"], sub_df["source_title"]))
assert all(ctx_law[q["context_id"]] == q["source_title"] for q in qa_all), "QA не соответствует parquet — перезапустите prepare_adilet_data.py"

if SMOKE_TEST:
    sub_df = sub_df.sample(n=120, random_state=SEED).sort_index()
    keep = set(sub_df["context_id"])
    qa_all = [q for q in qa_all if q["context_id"] in keep]

print(f"Фрагментов (контекстов): {len(sub_df)} | вопросов: {len(qa_all)}")

Фрагментов (контекстов): 973 | вопросов: 7443


## 2. Разбиение train / val / test по фрагментам

Внутри каждого закона фрагменты перемешиваются и делятся в пропорции `SPLIT_RATIOS`.
Так в каждой части представлены все 14 законов, а вопросы к одному фрагменту не «утекают» между частями.
Корпус для поиска при этом остаётся полным: при тестировании модель ищет среди всех чанков всех законов.

In [5]:
rng = np.random.default_rng(SEED)
split_of_ctx = {}
for law, group in sub_df.groupby("source_title"):
    ids = list(group["context_id"])
    rng.shuffle(ids)
    n_test = max(1, round(len(ids) * SPLIT_RATIOS[2]))
    n_val = max(1, round(len(ids) * SPLIT_RATIOS[1]))
    for j, ctx_id in enumerate(ids):
        split_of_ctx[ctx_id] = "test" if j < n_test else "val" if j < n_test + n_val else "train"

splits = {"train": [], "val": [], "test": []}
for q in qa_all:
    splits[split_of_ctx[q["context_id"]]].append(q)

# Ни один фрагмент не должен попасть в две части
ctx_sets = {name: {q["context_id"] for q in items} for name, items in splits.items()}
assert not (ctx_sets["train"] & ctx_sets["val"]) and not (ctx_sets["train"] & ctx_sets["test"]) and not (ctx_sets["val"] & ctx_sets["test"])

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
if not SMOKE_TEST:
    for name, items in splits.items():
        with open(SPLIT_DIR / f"qa_{LANG}_{name}.json", "w", encoding="utf-8") as f:
            json.dump(items, f, ensure_ascii=False, indent=2)
    print(f"Разбиение сохранено в {SPLIT_DIR}/qa_{LANG}_{{train,val,test}}.json")

stats = pd.DataFrame([
    {"Закон": q["source_title"], "Часть": name}
    for name, items in splits.items() for q in items
]).pivot_table(index="Закон", columns="Часть", aggfunc="size", fill_value=0)[["train", "val", "test"]]
stats.loc["ИТОГО"] = stats.sum()
stats

Разбиение сохранено в data\adilet\splits/qa_rus_{train,val,test}.json


Часть,train,val,test
Закон,,,
Бюджетный кодекс РК,1101,137,132
Гражданский кодекс РК (Общая часть),867,111,111
Закон РК О банках и банковской деятельности,115,16,13
Закон РК О воинской службе и статусе военнослужащих,691,95,90
Закон РК О долевом участии в жилищном строительстве,149,15,16
Закон РК О дорожном движении,43,7,8
Закон РК О пенсионном обеспечении,83,8,8
Закон РК О хозяйственных товариществах,1016,133,129
Закон РК Об образовании,568,74,72


## 3. Чанкинг корпуса и эталонные чанки

Фрагменты датасета длиной ~1100 токенов не помещаются в лимит e5 (512 токенов), поэтому обучаем и ищем на чанках по 500 символов.
Ответы в датасете — пересказ закона, а не цитата, поэтому точное совпадение подстроки находится редко (~20% вопросов).
Эталонные чанки ищутся внутри фрагмента вопроса в три шага (`GoldChunkMatcher` в `run_adilet_eval.py`):
1. **exact** — чанк содержит начало ответа (`relevant_text`), как в `run_adilet_eval.py`;
2. **overlap** — чанк с наибольшей долей основ слов ответа (первые 6 букв слова), если доля ≥ `MIN_COVERAGE`.
   Чанки, уступающие лучшему не больше `TIE_MARGIN`, тоже считаются эталонными: при почти равной доле
   по словам не определить, в каком из них ответ (часто он лежит на границе двух чанков). При выборе одного
   чанка в таких случаях ручная проверка выборки находила неверный эталон примерно в каждом втором;
3. **context** — иначе эталоном считаются все чанки фрагмента. Такие вопросы участвуют в оценке, но не в обучении:
   положительный пример должен быть точным.

Из-за шага 2 метрики уровня чанков строже, чем в `run_adilet_eval.py`, и напрямую с ним не сравниваются.

In [6]:
chunks, chunk_ids, chunk_ctx = build_fixed_chunks(sub_df, CHUNK_SIZE, CHUNK_OVERLAP)
matcher = GoldChunkMatcher(chunks, chunk_ctx, min_coverage=MIN_COVERAGE, tie_margin=TIE_MARGIN)

for items in splits.values():
    for q in items:
        q["gold_idx"], q["gold_type"] = matcher.find(q)

print(f"Чанков в корпусе: {len(chunks)}")
gold_stats = pd.DataFrame({
    name: pd.Series([q["gold_type"] for q in items]).value_counts(normalize=True)
    for name, items in splits.items()
}).reindex(["exact", "overlap", "context"]).fillna(0)
gold_stats.loc["overlap: несколько эталонов"] = [
    np.mean([q["gold_type"] == "overlap" and len(q["gold_idx"]) > 1 for q in items]) for items in splits.values()
]
gold_stats.style.format("{:.1%}")

Чанков в корпусе: 10402


,train,val,test
exact,22.1%,23.7%,24.2%
overlap,73.3%,70.1%,69.3%
context,4.6%,6.1%,6.5%
overlap: несколько эталонов,9.1%,8.4%,9.8%


## 4. Функция оценки

Метрики считаются на уровне чанков (MRR@10, HitRate@1/3/10, Recall@10) и на уровне фрагментов
(`ctx_Hit@k` — найден ли нужный фрагмент закона, сопоставимо со стратегией «контексты целиком»).

In [7]:
EVAL_K = 10
bm25 = BM25Retriever()
bm25.add_documents(chunks, chunk_ids)
chunk_pos = {cid: i for i, cid in enumerate(chunk_ids)}


def ir_metrics(items, predicted_ids):
    gold = [[chunk_ids[i] for i in q["gold_idx"]] for q in items]
    # Уровень фрагментов: убираем повторы фрагмента, сохраняя порядок выдачи
    pred_ctx = [list(dict.fromkeys(chunk_ctx[chunk_pos[c]] for c in row)) for row in predicted_ids]
    gold_ctx = [[q["context_id"]] for q in items]
    return {
        "MRR@10": calculate_mrr(gold, predicted_ids),
        "Hit@1": calculate_hit_rate_at_k(gold, predicted_ids, k=1),
        "Hit@3": calculate_hit_rate_at_k(gold, predicted_ids, k=3),
        "Hit@10": calculate_hit_rate_at_k(gold, predicted_ids, k=10),
        "Recall@10": calculate_recall_at_k(gold, predicted_ids, k=10),
        "ctx_Hit@1": calculate_hit_rate_at_k(gold_ctx, pred_ctx, k=1),
        "ctx_Hit@3": calculate_hit_rate_at_k(gold_ctx, pred_ctx, k=3),
    }


def build_dense_index(model_name_or_path):
    # В ключ кэша входит отпечаток файлов весов (model_cache_name), поэтому векторы
    # переобученной модели не перепутаются со старыми
    embedder = EmbeddingModel(str(model_name_or_path), device=DEVICE)
    chunk_emb = get_or_compute_embeddings(
        embedder, chunks, CACHE_DIR,
        cache_key=f"{model_cache_name(str(model_name_or_path))}_{LANG}_fixed_{CHUNK_SIZE}_{CHUNK_OVERLAP}",
    )
    index = VectorRetriever(embedding_dim=chunk_emb.shape[1])
    index.add_documents(chunk_emb, chunks, chunk_ids)
    return embedder, index


def evaluate(embedder, index, items):
    # Гибрид здесь с параметрами по умолчанию (0.5 / 0.5, rrf_k=60); подобранные на val — в tune_hybrid.py
    queries = [q["query"] for q in items]
    q_emb = embedder.encode_queries(queries)
    _, idx = index.search(q_emb, k=EVAL_K)
    dense_pred = [[chunk_ids[i] for i in row if i >= 0] for row in idx]
    hybrid_pred = HybridRetriever(index, bm25).batch_search(queries, q_emb, k=EVAL_K)
    return {"dense": ir_metrics(items, dense_pred), "hybrid": ir_metrics(items, hybrid_pred)}, dense_pred


def evaluate_bm25(items):
    return ir_metrics(items, bm25.batch_search([q["query"] for q in items], k=EVAL_K))

## 5. Baseline: модель без дообучения

In [8]:
base_embedder, base_index = build_dense_index(BASE_MODEL)

results = {"bm25": {}, "base": {}, "finetuned": {}, "finetuned_std": {}}
base_dense_pred = {}
for name in ["val", "test"]:
    results["bm25"][name] = evaluate_bm25(splits[name])
    results["base"][name], base_dense_pred[name] = evaluate(base_embedder, base_index, splits[name])

pd.DataFrame({
    "BM25": results["bm25"]["test"],
    "Dense (base)": results["base"]["test"]["dense"],
    "Hybrid (base)": results["base"]["test"]["hybrid"],
}).T.round(4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

⚡ Загрузка векторов из дискового кэша: data/adilet/cache\intfloat_multilingual-e5-small_rus_fixed_500_50_239851b5c576e70d.npy...
  Успешно загружено 10402 векторов за 0.02 сек. Форма: (10402, 384)


,MRR@10,Hit@1,Hit@3,Hit@10,Recall@10,ctx_Hit@1,ctx_Hit@3
BM25,0.5375,0.4450,0.6147,0.7042,0.6484,0.5875,0.7300
Dense (base),0.5184,0.4206,0.5807,0.7327,0.6758,0.5834,0.7476
Hybrid (base),0.5653,0.4654,0.6459,0.7558,0.6970,0.6214,0.7748


## 6. Обучающие тройки и hard negatives

Для каждого train-вопроса: `(вопрос, эталонный чанк, сложный негатив)`.
Негатив — чанк, который исходная модель считает похожим на вопрос, но который:
- взят из **другого** фрагмента (соседние чанки того же фрагмента могут содержать часть ответа);
- не дублирует текст эталона;
- похож на вопрос заметно меньше эталона (`sim < NEG_MAX_RATIO * sim_pos`) — отсекает вероятные ложные негативы (подход NV-Retriever).

In [9]:
train_items = [q for q in splits["train"] if q["gold_type"] != "context"]
train_q_emb = base_embedder.encode_queries([q["query"] for q in train_items])
neg_scores, neg_idx = base_index.search(train_q_emb, k=NEG_SEARCH_DEPTH)

chunk_idx_by_law = defaultdict(list)
for i, ctx_id in enumerate(chunk_ctx):
    chunk_idx_by_law[ctx_law[ctx_id]].append(i)

triples = []
fallback = 0
for q, q_vec, scores, row in zip(train_items, train_q_emb, neg_scores, neg_idx):
    pos = q["gold_idx"][0]
    pos_score = float(q_vec @ base_index.index.reconstruct(pos))
    candidates = [
        i for i, s in zip(row, scores)
        if i >= 0 and chunk_ctx[i] != q["context_id"] and chunks[i] != chunks[pos] and s < NEG_MAX_RATIO * pos_score
    ][:NEG_POOL]
    if candidates:
        neg = int(rng.choice(candidates))
    else:
        # Подходящих кандидатов нет: берём случайный чанк того же закона из другого фрагмента
        fallback += 1
        same_law = [i for i in chunk_idx_by_law[q["source_title"]] if chunk_ctx[i] != q["context_id"]]
        pool = same_law or [i for i in range(len(chunks)) if chunk_ctx[i] != q["context_id"]]
        neg = int(rng.choice(pool))
    triples.append({"anchor": q["query"], "positive": chunks[pos], "negative": chunks[neg]})

print(f"Обучающих троек: {len(triples)} (случайный негатив: {fallback})")
example = triples[0]
print("\nВопрос:  ", example["anchor"])
print("Эталон:  ", example["positive"][:200], "...")
print("Негатив: ", example["negative"][:200], "...")

Обучающих троек: 5685 (случайный негатив: 1564)

Вопрос:   Когда был принят Земельный кодекс Республики Казахстан и под каким номером?
Эталон:   Земельный кодекс Республики Казахстан - ИПС "Әділет" 

Земельный кодекс Республики Казахстан

Обновленный 

Кодекс Республики Казахстан от 20 июня 2003 года № 442.

СОДЕРЖАНИЕ 

Примечание ИЗПИ! 

Для ...
Негатив:  лики Казахстан. 

7. Право землепользования других государств на территории Республики Казахстан возникает в соответствии с международными договорами, ратифицированными Республикой Казахстан. 

Сноска ...


## 7. Дообучение

- Функция потерь `MultipleNegativesRankingLoss`: для каждого вопроса эталон должен быть ближе, чем hard negative
  и чанки остальных вопросов батча (in-batch negatives).
- `BatchSamplers.NO_DUPLICATES` не кладёт в один батч одинаковые тексты: у одного чанка бывает несколько вопросов,
  и без этого эталон одного вопроса стал бы ложным негативом для другого.
- Префиксы `query: ` / `passage: ` добавляются вручную, как при инференсе в `src/embeddings.py`.
- После каждой эпохи считается MRR@10 на val; в конце загружается лучшая эпоха.
- Обучение повторяется для каждого seed из `TRAIN_SEEDS` на одних и тех же тройках. Модель каждого запуска
  сохраняется в `CHECKPOINT_DIR/seed_N`, оценивается на val и test, а в `OUTPUT_DIR` (её используют `ask.py` и демо)
  копируется запуск с лучшим MRR@10 на **val** — test в выборе не участвует.
- Цикл возобновляемый: если модель запуска уже сохранена в `CHECKPOINT_DIR/seed_N/model`, она не обучается заново,
  а только оценивается. Чтобы переобучить запуск, удалите его папку.
- Исходная модель выгружается из видеопамяти перед обучением, а после каждого запуска память освобождается явно:
  на GPU с 4 ГБ иначе третий запуск не помещается.

In [10]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer import losses
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator
from sentence_transformers.sentence_transformer.training_args import BatchSamplers


def free_gpu():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


# Исходная модель больше не нужна (негативы найдены, её выдача сохранена в base_dense_pred) — освобождаем видеопамять
del base_embedder
free_gpu()

val_evaluator = InformationRetrievalEvaluator(
    queries={q["id"]: f"query: {q['query']}" for q in splits["val"]},
    corpus={cid: f"passage: {text}" for cid, text in zip(chunk_ids, chunks)},
    relevant_docs={q["id"]: {chunk_ids[i] for i in q["gold_idx"]} for q in splits["val"]},
    name="val",
    mrr_at_k=[10],
    ndcg_at_k=[10],
    accuracy_at_k=[1, 3, 10],
    precision_recall_at_k=[10],
    batch_size=64,
)

runs = {}
for train_seed in TRAIN_SEEDS:
    print(f"\n===== seed {train_seed} =====")
    run_dir = CHECKPOINT_DIR / f"seed_{train_seed}"
    model_dir = run_dir / "model"

    if (model_dir / "config.json").exists():
        print(f"Модель уже обучена: {model_dir} — только оценка")
    else:
        if run_dir.exists():
            shutil.rmtree(run_dir)  # остатки прерванного запуска
        model = SentenceTransformer(BASE_MODEL, device=DEVICE)
        model.max_seq_length = MAX_SEQ_LEN

        train_dataset = Dataset.from_dict({
            "anchor": [f"query: {t['anchor']}" for t in triples],
            "positive": [f"passage: {t['positive']}" for t in triples],
            "negative": [f"passage: {t['negative']}" for t in triples],
        }).shuffle(seed=train_seed)

        args = SentenceTransformerTrainingArguments(
            output_dir=str(run_dir),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            warmup_steps=WARMUP_RATIO,  # в Transformers v5 дробное значение означает долю шагов
            fp16=DEVICE == "cuda",
            batch_sampler=BatchSamplers.NO_DUPLICATES,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="eval_val_cosine_mrr@10",
            greater_is_better=True,
            logging_steps=20,
            seed=train_seed,
            report_to="none",
        )
        trainer = SentenceTransformerTrainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            loss=losses.MultipleNegativesRankingLoss(model),
            evaluator=val_evaluator,
        )
        trainer.train()

        model.save(str(model_dir))
        pd.DataFrame(trainer.state.log_history).to_csv(run_dir / "history.csv", index=False)
        # Промежуточные чекпойнты (с состоянием оптимизатора) больше не нужны — освобождаем диск
        for ckpt in run_dir.glob("checkpoint-*"):
            shutil.rmtree(ckpt)

        # Освобождаем видеопамять: иначе векторизация идёт в десятки раз медленнее, а следующий запуск не помещается
        trainer.accelerator.free_memory()
        model.to("cpu")
        del trainer, model, train_dataset
        free_gpu()

    embedder, index = build_dense_index(model_dir)
    run = {"val": {}, "test": {}, "dense_pred": {}}
    for name in ["val", "test"]:
        run[name], run["dense_pred"][name] = evaluate(embedder, index, splits[name])
    runs[train_seed] = run
    print(f"seed {train_seed}: MRR@10 dense val {run['val']['dense']['MRR@10']:.4f} | test {run['test']['dense']['MRR@10']:.4f}")
    del embedder, index
    free_gpu()

best_seed = max(runs, key=lambda s: runs[s]["val"]["dense"]["MRR@10"])
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
shutil.copytree(CHECKPOINT_DIR / f"seed_{best_seed}" / "model", OUTPUT_DIR)
print(f"\nЛучший по val запуск: seed {best_seed} → скопирован в {OUTPUT_DIR}")


===== seed 42 =====
Модель уже обучена: checkpoints\e5-small-adilet-rus\seed_42\model — только оценка


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

⚡ Загрузка векторов из дискового кэша: data/adilet/cache\checkpoints_e5-small-adilet-rus_seed_42_model_a2f81fde_rus_fixed_500_50_239851b5c576e70d.npy...
  Успешно загружено 10402 векторов за 0.01 сек. Форма: (10402, 384)
seed 42: MRR@10 dense val 0.6451 | test 0.6241

===== seed 43 =====
Модель уже обучена: checkpoints\e5-small-adilet-rus\seed_43\model — только оценка


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

⚡ Загрузка векторов из дискового кэша: data/adilet/cache\checkpoints_e5-small-adilet-rus_seed_43_model_093ddd5d_rus_fixed_500_50_239851b5c576e70d.npy...
  Успешно загружено 10402 векторов за 0.02 сек. Форма: (10402, 384)
seed 43: MRR@10 dense val 0.6556 | test 0.6241

===== seed 44 =====


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Векторизация 10402 текстов через нейросеть...


Batches:   0%|          | 0/326 [00:00<?, ?it/s]

Векторизация завершена за 22.57 сек. Форма матрицы: (10402, 384)
💾 Векторы сохранены в кэш: data/adilet/cache\checkpoints_e5-small-adilet-rus_seed_44_model_62a1cfa6_rus_fixed_500_50_239851b5c576e70d.npy (15.2 MB)
seed 44: MRR@10 dense val 0.6557 | test 0.6256

Лучший по val запуск: seed 44 → скопирован в models\e5-small-adilet-rus


In [11]:
# Динамика обучения: MRR@10 на val после каждой эпохи (журнал оценщика) и средний loss эпохи (если сохранён)
dynamics = {}
for train_seed in runs:
    run_dir = CHECKPOINT_DIR / f"seed_{train_seed}"
    val_log = pd.read_csv(run_dir / "eval" / "Information-Retrieval_evaluation_val_results.csv")
    dynamics[(train_seed, "val MRR@10")] = val_log.set_index(val_log["epoch"].round().astype(int))["cosine-MRR@10"]
    if (run_dir / "history.csv").exists():
        h = pd.read_csv(run_dir / "history.csv").dropna(subset=["loss"])
        dynamics[(train_seed, "loss")] = h.groupby(np.ceil(h["epoch"]).astype(int))["loss"].mean()
pd.DataFrame(dynamics).rename_axis("эпоха").sort_index(axis=1).round(4)

42         43      44           
      val MRR@10 val MRR@10    loss val MRR@10
эпоха                                         
1         0.6425     0.6432  0.7246     0.6440
2         0.6457     0.6558  0.2816     0.6558

## 8. Сравнение «до / после» на тестовой выборке

In [12]:
METRICS = list(results["bm25"]["test"])


def aggregate(split, method):
    values = pd.DataFrame([runs[s][split][method] for s in runs])
    return values.mean().to_dict(), values.std(ddof=0).to_dict()


for name in ["val", "test"]:
    results["finetuned"][name], results["finetuned_std"][name] = {}, {}
    for method in ["dense", "hybrid"]:
        results["finetuned"][name][method], results["finetuned_std"][name][method] = aggregate(name, method)
results["finetuned_runs"] = {s: {name: runs[s][name] for name in ["val", "test"]} for s in runs}


def fmt(mean, std=None):
    return {m: f"{mean[m]:.4f}" + (f" ± {std[m]:.4f}" if std is not None else "") for m in METRICS}


ft, ft_std = results["finetuned"]["test"], results["finetuned_std"]["test"]
comparison = pd.DataFrame({
    "BM25": fmt(results["bm25"]["test"]),
    "Dense (base)": fmt(results["base"]["test"]["dense"]),
    f"Dense (fine-tuned, {len(runs)} seed)": fmt(ft["dense"], ft_std["dense"]),
    "Hybrid (base)": fmt(results["base"]["test"]["hybrid"]),
    f"Hybrid (fine-tuned, {len(runs)} seed)": fmt(ft["hybrid"], ft_std["hybrid"]),
}).T
print(f"Test, {len(splits['test'])} вопросов; для дообученной модели — среднее ± стандартное отклонение по seed {list(runs)}")
comparison

Test, 737 вопросов; для дообученной модели — среднее ± стандартное отклонение по seed [42, 43, 44]


,MRR@10,Hit@1,Hit@3,Hit@10,Recall@10,ctx_Hit@1,ctx_Hit@3
BM25,0.5375,0.4450,0.6147,0.7042,0.6484,0.5875,0.7300
Dense (base),0.5184,0.4206,0.5807,0.7327,0.6758,0.5834,0.7476
"Dense (fine-tuned, 3 seed)",0.6246 ± 0.0007,0.5152 ± 0.0039,0.7151 ± 0.0019,0.8241 ± 0.0013,0.7638 ± 0.0010,0.6807 ± 0.0028,0.8250 ± 0.0033
Hybrid (base),0.5653,0.4654,0.6459,0.7558,0.6970,0.6214,0.7748
"Hybrid (fine-tuned, 3 seed)",0.6338 ± 0.0015,0.5414 ± 0.0033,0.7001 ± 0.0033,0.8213 ± 0.0039,0.7534 ± 0.0017,0.6834 ± 0.0036,0.8209 ± 0.0029


In [13]:
# Примеры вопросов, где исходная модель промахнулась в топ-3, а дообученная (лучший запуск) нашла эталон на 1-м месте
test_items = splits["test"]
ft_dense_test = runs[best_seed]["dense_pred"]["test"]
improved = [
    (q, b, f) for q, b, f in zip(test_items, base_dense_pred["test"], ft_dense_test)
    if not {chunk_ids[i] for i in q["gold_idx"]} & set(b[:3]) and f[:1] and chunk_pos[f[0]] in q["gold_idx"]
]
base_hit1 = sum(bool(b[:1]) and chunk_pos[b[0]] in q["gold_idx"] for q, b in zip(test_items, base_dense_pred["test"]))
ft_hit1 = sum(bool(f[:1]) and chunk_pos[f[0]] in q["gold_idx"] for q, f in zip(test_items, ft_dense_test))
print(f"Эталон на 1-м месте: base {base_hit1}, fine-tuned {ft_hit1} из {len(test_items)} (разница {ft_hit1 - base_hit1:+d})")
print(f"Из них base не было даже в топ-3, а fine-tuned поставила на 1-е место: {len(improved)}\n")
for q, b, f in improved[:5]:
    print(f"[{q['source_title']}] {q['query']}")
    print(f"  base топ-1:       {chunks[chunk_pos[b[0]]][:150]}...")
    print(f"  fine-tuned топ-1: {chunks[chunk_pos[f[0]]][:150]}...\n")

Эталон на 1-м месте: base 310, fine-tuned 383 из 737 (разница +73)
Из них base не было даже в топ-3, а fine-tuned поставила на 1-е место: 48

[Земельный кодекс РК] Что такое изъятие согласно данному Кодексу?
  base топ-1:       естре лиц, у которых принудительно изъяты земельные участки, исключаются по истечении трех лет со дня вступления в законную силу решения суда о принуд...
  fine-tuned топ-1: г и выдаче их результатов услугополучателю по принципу "одного окна", обеспечения оказания государственных услуг в электронной форме;

1-1) изъятие – ...

[Земельный кодекс РК] Кто может быть аффилиированным лицом?
  base топ-1:       мера уставного капитала), сведения об аффилированных лицах заявителя;...
  fine-tuned топ-1: - резервные земли, образуемые за счет угодий сельскохозяйственного назначения, а также земельных участков, не используемых по назначению либо использу...

[Земельный кодекс РК] В какой срок вступил в силу закон, внесший изменения в Статью 14 от 06.01.2011 № 378‑IV?
  ba

In [14]:
if not SMOKE_TEST:
    report = {
        "language": LANG,
        "base_model": BASE_MODEL,
        "finetuned_model": OUTPUT_DIR.as_posix(),
        "seed": SEED,
        "train_seeds": TRAIN_SEEDS,
        "best_seed_by_val": best_seed,
        "split_ratios": SPLIT_RATIOS,
        "split_sizes": {name: len(items) for name, items in splits.items()},
        "gold_types": {name: pd.Series([q["gold_type"] for q in items]).value_counts().to_dict() for name, items in splits.items()},
        "train_triples": len(triples),
        "hyperparams": {
            "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
            "warmup_ratio": WARMUP_RATIO, "max_seq_len": MAX_SEQ_LEN,
            "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP,
            "min_coverage": MIN_COVERAGE, "tie_margin": TIE_MARGIN,
        },
        # finetuned — среднее по запускам, finetuned_std — стандартное отклонение, finetuned_runs — каждый запуск
        "results": results,
    }
    report_path = Path(f"data/adilet/finetune_results_{LANG}.json")
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(f"Отчёт сохранён: {report_path}")

Отчёт сохранён: data\adilet\finetune_results_rus.json


## Дальнейшие шаги

- Подобрать веса гибрида на val для сохранённой модели: `python tune_hybrid.py --lang rus`.
- Проверить модель в поиске: `python ask.py` или `streamlit run app.py` (дообученная модель выбирается по умолчанию).
- Запустить ноутбук с `LANG = "kaz"`.
- Сравнить с `intfloat/multilingual-e5-base` (поменять `BASE_MODEL`, уменьшить `BATCH_SIZE` при нехватке памяти).